# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring** (predefined Lane 2).

I'm picking this lane because the starter dataset already ships everything the lane needs: 90-day observed signals (impressions, clicks, sessions, CTR, average position), a trend bucket, and freshness/age fields — and the starter pipeline in this repo already builds a transparent baseline score and a model that beats it (random forest Precision@50 of 0.740 vs. 0.240 for the baseline rules, per the lane guide). That gap tells me there is real, learnable signal in this data for *ranking pages by review priority*, not just noise. It's also the most directly useful output for FlyRank's actual workflow: a content team has limited review capacity, so turning raw signals into an ordered queue with reason codes is something a content strategist could act on in week one, and something I can keep sharpening (better labels, better validation) for the rest of the internship.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Given a content page's observed 90-day search and engagement signals, which pages should a content reviewer look at first — and with which suggested action (refresh, protect, monitor, or prune)?

- **Unit of analysis:** one content page (`content_id`), scored using its trailing 90-day window of observed signals.
- **Decision it improves:** which pages a content team spends its limited weekly review time on, out of a large inventory, and what kind of action (refresh vs. leave alone) to try first.
- **Who acts on it:** a content strategist or SEO reviewer with a fixed weekly capacity (e.g., they can only realistically review 20–50 pages a week).
- **Cost of a wrong call:**
  - *False positive* (flagged as high-priority but not actually worth it): wastes a reviewer's limited hours on a page that wasn't really declining, or was declining for reasons a refresh can't fix (e.g., seasonality or consolidation) — that's real opportunity cost against the pages that actually needed attention.
  - *False negative* (a genuinely declining, high-demand page never surfaces): visibility and traffic keep eroding silently until someone notices manually, which is slower and more expensive than catching it early.
- **Why ML/data helps at all:** with tens of thousands of pages and only a handful of reviewers, someone has to rank the queue somehow. The starter baseline already shows a hand-written rule (`baseline_refresh_score`) gets only ~24% precision in its top 50; a model trained on the same observed signals gets to ~74% in this repo's own results. That's the case for this not being just 'train a model for its own sake' — the model is a ranking tool that measurably beats the status quo rule at the one thing that matters operationally: putting real opportunities near the top of a capacity-limited list.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Apply the same eligibility filter the starter pipeline uses (see lane guide section 5):
# impressions_90d > 0 and content_age_days >= 90, deduplicated by content_id.
elig = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")

n_total = len(df)
n_elig = len(elig)
n_clients = elig["client_id"].nunique()

pct_declining = 100 * (elig["trend_direction"] == "down").mean()

declining_with_demand = elig[(elig["trend_direction"] == "down") & (elig["impressions_90d"] >= 100)]
pct_declining_with_demand = 100 * len(declining_with_demand) / n_elig

low_ctr = elig[
    (elig["impressions_90d"] >= 500)
    & (elig["avg_position"] > 0) & (elig["avg_position"] <= 20)
    & (elig["ctr"] < 0.5)
]
pct_low_ctr = 100 * len(low_ctr) / n_elig

print(f"Eligible rows (impressions_90d > 0, age >= 90d): {n_elig:,} of {n_total:,} total, across {n_clients} clients")
print(f"1) Share of eligible pages currently trending down: {pct_declining:.1f}%")
print(f"2) Share flagged 'declining_with_demand' (down AND >=100 impressions/90d): {pct_declining_with_demand:.1f}% ({len(declining_with_demand):,} pages)")
print(f"3) Share flagged 'low_ctr_visible_page' (>=500 impressions, position 1-20, CTR<0.5): {pct_low_ctr:.1f}% ({len(low_ctr):,} pages)")


Eligible rows (impressions_90d > 0, age >= 90d): 30,000 of 30,000 total, across 32 clients
1) Share of eligible pages currently trending down: 54.2%
2) Share flagged 'declining_with_demand' (down AND >=100 impressions/90d): 43.8% (13,152 pages)
3) Share flagged 'low_ctr_visible_page' (>=500 impressions, position 1-20, CTR<0.5): 32.5% (9,759 pages)


**What these numbers say:** over half of eligible pages (54.2%) are currently trending down, and 43.8% combine a downward trend with real demand (100+ impressions in 90 days) — that's over 13,000 pages, far more than any reviewer could manually triage. On top of that, 32.5% of eligible pages are ranking on page 1–2 (position ≤ 20) with meaningful traffic but a CTR under 0.5%, meaning they're visible but under-capturing clicks — a different, cheaper fix (metadata/snippet) than a full content refresh. With three overlapping but distinct opportunity segments this large, a simple ranked queue with reason codes is clearly worth building: there's no way a human could eyeball 30,000 rows and consistently pick the right 20–50 to review first without some scoring system behind it.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [2]:
claims = {
    "I CAN say": [
        "These pages showed an observed downward trend in impressions/clicks over the measured 90-day window.",
        "This ranking is directional/decision-support: it orders pages by evidence of risk or opportunity so a reviewer spends limited time on the most promising candidates first.",
        "A learned model, validated on held-out clients, ranked candidates more precisely (higher Precision@K) than a fixed hand-written rule on this starter slice.",
        "A page's low CTR at a good average position is an observed mismatch between visibility and clicks — worth a human look, not proof of any single cause.",
    ],
    "I CANNOT say": [
        "That a refresh CAUSED any recovery (that needs an experiment or causal design, not this data).",
        "That I 'predicted Google's algorithm' or discovered a ranking factor — I only found associations in observable signals.",
        "That a 'declining' label guarantees the page is actually declining rather than experiencing consolidation, seasonality, or noise (section 7 of the lane guide) unless I've explicitly ruled those out.",
        "That results from this 30,000-row anonymized starter slice generalize to the full ~79M-row warehouse without re-validating there.",
    ],
}

for header, items in claims.items():
    print(header + ":")
    for item in items:
        print(f"  - {item}")
    print()


I CAN say:
  - These pages showed an observed downward trend in impressions/clicks over the measured 90-day window.
  - This ranking is directional/decision-support: it orders pages by evidence of risk or opportunity so a reviewer spends limited time on the most promising candidates first.
  - A learned model, validated on held-out clients, ranked candidates more precisely (higher Precision@K) than a fixed hand-written rule on this starter slice.
  - A page's low CTR at a good average position is an observed mismatch between visibility and clicks — worth a human look, not proof of any single cause.

I CANNOT say:
  - That a refresh CAUSED any recovery (that needs an experiment or causal design, not this data).
  - That I 'predicted Google's algorithm' or discovered a ranking factor — I only found associations in observable signals.
  - That a 'declining' label guarantees the page is actually declining rather than experiencing consolidation, seasonality, or noise (section 7 of the lane 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.